# ShelfWise - Full-application stress test under the world simulation

**Run every cell top to bottom (Kernel -> Restart & Run All).** This does not test the fine-tuned
model in isolation - it drives the *whole running application* the way it would actually be used:
the world simulation (`src/shelfwise_worldgen/`) generates fresh synthetic retail events across a
realistic, full-supermarket product assortment (thousands of SKUs spanning dozens of departments -
produce, dairy, electronics, pharmacy, alcohol, everything - not just the one hero SKU), every event
goes through the real ingest -> cascade -> Critic -> Executive -> HITL pipeline, and periodically a
natural-language question about whatever the simulation just generated is sent through the real chat
endpoint, so the app is being exercised end to end, not just the model.

**Important correction from earlier guidance:** the deterministic decision-science cascade (the part
the world simulation drives) never calls any LLM at all, by design - the math is meant to be
auditable and testable on its own, not guessed by a model. The **only** place this app actually calls
an LLM is the chat feature, and it always calls the "strong" model tier. That means the environment
variable that actually matters for exercising your fine-tuned model here is **`LLM_STRONG_MODEL`**,
not `LLM_ROUTINE_MODEL` as suggested in the fine-tune notebook's final instructions.

**Before running, in a terminal in this same pod:**
1. Make sure `vllm serve ... --enable-lora --lora-modules shelfwise=<adapter_dir> ...` (printed at
   the end of `02_shelfwise_gemma_finetune.ipynb`) is already running.
2. Set these two environment variables **before starting this notebook's kernel**:
   - `LLM_BASE_URL=http://127.0.0.1:8000`
   - `LLM_STRONG_MODEL=shelfwise`

If you skip that, this notebook still runs correctly and still stress-tests the whole application -
chat answers just come from the deterministic offline fallback instead of your fine-tuned model. This
notebook checks for and reports that explicitly, it does not fail silently.

Set these to control scope:

- `STRESS_HOURS` (default `3.0`) - wall-clock budget for the whole stress loop.
- `STRESS_ASSORTMENT_SIZE` (default `3000`) - how many products (out of the full generated catalog)
  make up the simulated store's assortment each cycle.
- `STRESS_CATALOG_SCALE` (default `supermarket`) - `convenience` (~1,035 possible products),
  `supermarket` (~7,254), or `hypermarket` (~24,675) - the pool `STRESS_ASSORTMENT_SIZE` draws from.
- `STRESS_CHAT_EVERY_N_CYCLES` (default `3`) - how often (in world-sim cycles) to also fire a sample
  chat question about a product from that cycle's assortment.

## 0. Setup

In [ ]:
from __future__ import annotations

import json
import os
import sys
from datetime import UTC, datetime
from pathlib import Path

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

STRESS_HOURS = float(os.environ.get("STRESS_HOURS", "3.0"))
STRESS_ASSORTMENT_SIZE = int(os.environ.get("STRESS_ASSORTMENT_SIZE", "3000"))
STRESS_CATALOG_SCALE = os.environ.get("STRESS_CATALOG_SCALE", "supermarket")
STRESS_CHAT_EVERY_N_CYCLES = int(os.environ.get("STRESS_CHAT_EVERY_N_CYCLES", "3"))
BASE_SEED = 20260709

print(f"repo root: {ROOT}")
print(f"stress hours: {STRESS_HOURS}")
print(f"assortment size: {STRESS_ASSORTMENT_SIZE} (scale: {STRESS_CATALOG_SCALE})")
print(f"chat sample every {STRESS_CHAT_EVERY_N_CYCLES} cycles")

RESULTS: list[dict] = []


def record(name: str, ok: bool, detail: str = "") -> None:
    RESULTS.append({"section": name, "ok": ok, "detail": detail})
    banner = "PASS" if ok else "FAIL"
    print(f"\n[{banner}] {name}" + (f" - {detail}" if detail else ""))


def run(name: str, fn) -> None:
    try:
        fn()
        record(name, True)
    except AssertionError as exc:
        record(name, False, f"assertion failed: {exc}")
        raise
    except Exception as exc:
        record(name, False, f"{type(exc).__name__}: {exc}")
        raise


## 1. Build the full-supermarket assortment (report coverage up front)

Uses `shelfwise_worldgen.catalog.sample.sample_assortment` directly - the same deterministic
generator the API uses internally - purely to report *how broad* this run's coverage actually is
before spending any time on it: how many departments, how many distinct product names, real examples
from outside the usual dairy/hero-SKU corner of the catalog.

In [ ]:
from shelfwise_worldgen.catalog.sample import sample_assortment


def _assortment_report() -> None:
    assortment = sample_assortment(BASE_SEED, size=STRESS_ASSORTMENT_SIZE, scale=STRESS_CATALOG_SCALE)
    departments = sorted({product.department for product in assortment})
    categories = sorted({product.category for product in assortment})

    print(f"assortment size: {len(assortment)}")
    print(f"departments covered: {len(departments)}")
    print(f"categories covered: {len(categories)}")
    print("sample departments:", departments[:15])

    non_food_hit = any(
        department in {"Electronics", "Pharmacy", "Alcohol", "Clothing", "Hardware"}
        for department in departments
    )
    print("touches non-food departments (electronics/pharmacy/alcohol/etc):", non_food_hit)

    assert len(assortment) == STRESS_ASSORTMENT_SIZE
    assert len(departments) > 10, "expected broad department coverage, not just a few categories"


run("Full-supermarket assortment coverage report", _assortment_report)


## 2. Check whether the fine-tuned model is actually wired in

This reports which inference path is configured. Offline mode is valid for a local integration run.
When `STRESS_REQUIRE_MODEL=1`, the full-system driver requires at least one observed model-backed
answer and raises if every chat response falls back offline.

In [ ]:
from shelfwise_inference import load_inference_config


def _chat_model_wiring_check() -> None:
    config = load_inference_config()
    print(json.dumps(config.to_public_dict(), indent=2))
    if config.provider.value == "offline":
        print(
            "\nchat will use the deterministic OFFLINE fallback, not a real model - "
            "set LLM_BASE_URL and LLM_STRONG_MODEL before restarting the kernel if you "
            "want this run to exercise your fine-tuned model."
        )
    else:
        print(f"\nchat will call: provider={config.provider.value} strong_model={config.strong_model}")


run("Chat model wiring check", _chat_model_wiring_check)


## 3. The stress loop - autonomous, no human in the loop

The `shelfwise_eval.full_system` driver rotates world conditions and explicitly receipts golden expiry, Critic rejection, procurement, sales, misprice, cold chain, connector duplicate/invalid handling, multimodal review, tenant isolation, worker retry/DLQ, HITL approval/rejection, write-back, learning, tools, event bus, inference, and observability. Every pending decision is resolved through the public HITL endpoints by the deterministic autopilot policy.

**Every decision leaves a trail**: each one is appended to `data/harness_runs/<run_id>/stress_test/decision_trail.jsonl` *as it happens* (crash-safe - an interruption in hour 7 keeps hours 1-6), with the cycle, scenario, autopilot verdict and reason, and the resolved status.

In [ ]:
import os
from collections import Counter

# Unattended soak runs push write rates far past interactive defaults; these knobs exist
# for exactly this. Must be set BEFORE the first shelfwise_backend.app import in this kernel
# (triggered transitively by the shelfwise_eval.full_system import below).
os.environ.setdefault("SHELFWISE_WRITE_RATE_CAPACITY", "1000000")
os.environ.setdefault("SHELFWISE_WRITE_RATE_REFILL_PER_S", "50000")

from shelfwise_eval.full_system import FullSystemConfig, run_full_system
from shelfwise_worldgen.scenarios import SCENARIOS


def _driver_stress_loop() -> None:
    run_id = os.environ.get("HARNESS_RUN_ID") or datetime.now(UTC).strftime("%Y%m%dT%H%M%SZ")
    run_dir = ROOT / "data" / "harness_runs" / run_id / "stress_test"
    sizes = tuple(
        int(item.strip())
        for item in os.environ.get("STRESS_ASSORTMENT_SIZES", "500,1500,3000").split(",")
        if item.strip()
    )
    scales = tuple(
        item.strip()
        for item in os.environ.get("STRESS_CATALOG_SCALES", "supermarket,hypermarket").split(",")
        if item.strip()
    )
    report = run_full_system(
        FullSystemConfig(
            base_seed=BASE_SEED,
            world_cycles=len(SCENARIOS),
            duration_seconds=STRESS_HOURS * 3600,
            assortment_sizes=sizes,
            catalog_scales=scales,
            event_limit=500,
            chat_every_n_cycles=STRESS_CHAT_EVERY_N_CYCLES,
            live_required=os.environ.get("STRESS_REQUIRE_MODEL") == "1",
            run_id=run_id,
            artifact_dir=run_dir,
        )
    )
    action_counts = Counter(
        str(row.get("action_type") or "unknown") for row in report.decision_trail
    )
    globals().update(
        FULL_SYSTEM_REPORT=report,
        RUN_ID=run_id,
        TRAIL_PATH=run_dir / "decision_trail.jsonl",
        STRESS_TOTALS=report.totals,
        STRESS_ACTION_COUNTS=dict(action_counts),
        STRESS_CHAT_SAMPLES=json.loads((run_dir / "chat_samples.json").read_text(encoding="utf-8")),
    )
    print(json.dumps(report.to_dict(), indent=2))
    report.require_passed()


run("Full-system world driver + strict audit gates", _driver_stress_loop)


## 4. Export this run's stress-test data

Same convention as `01_shelfwise_full_test_harness.ipynb`'s Section 9 - written to
`data/harness_runs/<run_id>/stress_test/` so it survives the kernel stopping, but still has to be
copied off this box before the environment is reclaimed (`data/harness_runs/` is gitignored on
purpose - raw generated telemetry does not belong in git history).

In [ ]:
def _export_stress_results() -> None:
    run_id = globals().get("RUN_ID") or datetime.now(UTC).strftime("%Y%m%dT%H%M%SZ")
    run_dir = ROOT / "data" / "harness_runs" / run_id / "stress_test"
    run_dir.mkdir(parents=True, exist_ok=True)

    totals = globals().get("STRESS_TOTALS", {})
    trail_path = globals().get("TRAIL_PATH")
    trail_lines = 0
    if trail_path is not None and Path(trail_path).exists():
        with Path(trail_path).open("r", encoding="utf-8") as handle:
            for line in handle:
                json.loads(line)
                trail_lines += 1

    from shelfwise_inference import load_inference_config

    latencies = sorted(globals().get("STRESS_CHAT_LATENCIES", []))

    def _pct(values: list[float], q: float) -> float | None:
        if not values:
            return None
        return values[min(int(len(values) * q), len(values) - 1)]

    manifest = {
        "trail_lines": trail_lines,
        "action_type_counts": globals().get("STRESS_ACTION_COUNTS", {}),
        "inference_config": load_inference_config().to_public_dict(),
        "chat_latency_ms": {
            "count": len(latencies),
            "p50": _pct(latencies, 0.50),
            "p95": _pct(latencies, 0.95),
            "max": latencies[-1] if latencies else None,
        },
        "run_id": run_id,
        "exported_at": datetime.now(UTC).isoformat(),
        "stress_hours_configured": STRESS_HOURS,
        "assortment_sizes": os.environ.get("STRESS_ASSORTMENT_SIZES", "500,1500,3000"),
        "catalog_scales": os.environ.get("STRESS_CATALOG_SCALES", "supermarket,hypermarket"),
        "totals": totals,
        "scenario_rotation": globals().get("STRESS_SCENARIO_COUNTS", {}),
        "cascade_scenarios": globals().get("STRESS_CASCADE_SCENARIOS", {}),
    }
    (run_dir / "manifest.json").write_text(json.dumps(manifest, indent=2, sort_keys=True), encoding="utf-8")
    (run_dir / "chat_samples.json").write_text(
        json.dumps(globals().get("STRESS_CHAT_SAMPLES", []), indent=2, sort_keys=True), encoding="utf-8"
    )

    from fastapi.testclient import TestClient

    from shelfwise_backend.app import app

    client = TestClient(app)
    learning = client.get("/learning").json()
    (run_dir / "learning_events.json").write_text(
        json.dumps(learning, indent=2, sort_keys=True), encoding="utf-8"
    )


    print(json.dumps(manifest, indent=2))
    print(f"\nwrote stress-test artifacts to: {run_dir}")
    assert (run_dir / "manifest.json").exists()
    assert trail_lines > 0, "decision trail must not be empty"


def _verify_driver_artifacts() -> None:
    report = globals().get("FULL_SYSTEM_REPORT")
    assert report is not None, "full-system report is missing"
    run_dir = Path(report.artifact_dir)
    required = {
        "manifest.json",
        "decision_trail.jsonl",
        "feature_receipts.json",
        "route_receipts.json",
        "learning_events.json",
        "chat_samples.json",
    }
    missing = sorted(name for name in required if not (run_dir / name).exists())
    assert not missing, f"missing full-system artifacts: {missing}"
    manifest = json.loads((run_dir / "manifest.json").read_text(encoding="utf-8"))
    trail = [
        json.loads(line)
        for line in (run_dir / "decision_trail.jsonl").read_text(encoding="utf-8").splitlines()
        if line.strip()
    ]
    assert manifest["passed"] is True, manifest["failures"]
    assert len(trail) == manifest["totals"]["decisions_total"]
    print(f"verified strict full-system artifacts at: {run_dir}")


run("Verify full-system artifacts on disk", _verify_driver_artifacts)


## Final summary

In [ ]:
print("=" * 72)
print("SHELFWISE STRESS TEST SUMMARY")
print("=" * 72)
width = max((len(item["section"]) for item in RESULTS), default=10)
failures = 0
for item in RESULTS:
    banner = "PASS" if item["ok"] else "FAIL"
    if not item["ok"]:
        failures += 1
    line = f"{banner:4}  {item['section']:<{width}}"
    if item["detail"]:
        line += f"  ({item['detail']})"
    print(line)
print("=" * 72)
if failures == 0:
    print("ALL CHECKS PASSED - the whole application held up under the full-catalog stress test.")
else:
    message = f"{failures} check(s) FAILED - scroll up to the matching section for the full output."
    print(message)
    raise RuntimeError(message)
